# Benchmarking `Baseline` — the minimal example

Everything you need to benchmark the `Baseline` knowledge-graph pipeline, as briefly as possible:

1. **Whole-pipeline** — run `Baseline` end-to-end on a tiny corpus and get an overall quality score.
2. **All six stage benchmarks** — `Dedup`, `Chunking`, `Extraction`, `Resolution`, `Quality`, `RAG` — one line each, against gold data.

> **If you just updated the library:** do **Kernel → Restart & Run All** — re-running cells alone keeps the old modules in memory. The library fails loudly (no silent fallbacks): if something breaks, it raises.

In [ ]:
# ── Setup: imports + tiny corpus ─────────────────────────────────
import json
from pathlib import Path

from polygraph._shared.stage_config import PreprocessConfig
from polygraph.benchmark_pipeline import Benchmark, BenchmarkRunner
from polygraph.pipelines import Baseline


def _find_root() -> Path:
    """Repo root = the directory holding pyproject.toml + src/polygraph."""
    for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (cand / "pyproject.toml").exists() and (cand / "src" / "polygraph").is_dir():
            return cand
    raise RuntimeError("Could not find the polygraph repo root above the working directory.")


REPO_ROOT = _find_root()
GOLD = REPO_ROOT / "benchmarks" / "data"
OUT = REPO_ROOT / "output" / "minimal"
OUT.mkdir(parents=True, exist_ok=True)

# A tiny 3-article corpus — enough to exercise every stage quickly.
CORPUS = OUT / "corpus.jsonl"
DOCS = [
    "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. "
    "She discovered the chemical elements radium and polonium in Paris, France. She won two Nobel "
    "prizes and remains the only person to win Nobel prizes in two different sciences. Her work on "
    "radioactivity was revolutionary and she is widely regarded as one of the greatest scientists of all time.",
    "Albert Einstein was a theoretical physicist who developed the theory of relativity. He was born "
    "in Ulm, Germany in 1879 and later emigrated to the United States. He won the Nobel Prize in "
    "Physics for his work on the photoelectric effect. He is best known for the mass energy "
    "equivalence formula and spent his later years at Princeton University working on unified field theory.",
    "Alan Turing was a British mathematician and computer scientist who laid the foundations of "
    "modern computing. He formalised the concept of the Turing machine and made pivotal contributions "
    "to code-breaking during the second world war. His work on artificial intelligence and computability "
    "shaped the field for decades and continues to influence computer science today.",
]
CORPUS.write_text("\n".join(json.dumps({"text": d}) for d in DOCS) + "\n", encoding="utf-8")


def gold(name: str, n: int | None = None) -> Path:
    """Gold dataset path; when n is given, keep only the first n records."""
    if n is None:
        return GOLD / name
    out = OUT / f"sample_{name}"
    with (GOLD / name).open(encoding="utf-8") as f:
        out.write_text("".join(next(f) for _ in range(n)), encoding="utf-8")
    return out


print("corpus:", CORPUS)

In [ ]:
# ── 1) Whole-pipeline benchmark ───────────────────────────────────
# Run the full Baseline pipeline on the corpus and get one quality score.
# sentence chunking + minhash dedup = fast, no embedding model needed.
runner = BenchmarkRunner(
    pipeline=Baseline,
    input_paths=[str(CORPUS)],
    output_dir=str(OUT / "whole"),
    extra={
        "preprocess": PreprocessConfig(
            chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
        )
    },
)
r = runner.run()
print(
    f"overall_score={r.overall_score:.3f}  nodes={r.metrics['num_nodes']}  edges={r.metrics['num_edges']}"
)
print("report:", Path(r.output_dir) / "results_summary.json")

In [ ]:
# ── 2) All six stage benchmarks, one line each ────────────────────
# RAG needs a self-contained gold (the HotpotQA host is usually offline).
RAG_GOLD = OUT / "rag_gold.jsonl"
RAG_GOLD.write_text(
    "\n".join(
        json.dumps(q)
        for q in [
            {
                "query": "Who was Albert Einstein?",
                "answer_entity": "Albert Einstein",
                "supporting_chunks": [
                    "Albert Einstein was a theoretical physicist who developed the theory of relativity."
                ],
            },
            {
                "query": "Who was Marie Curie?",
                "answer_entity": "Marie Curie",
                "supporting_chunks": [
                    "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity."
                ],
            },
            {
                "query": "What is the Turing machine?",
                "answer_entity": "the Turing machine",
                "supporting_chunks": [
                    "He formalised the concept of the Turing machine and made pivotal contributions to code-breaking during the second world war."
                ],
            },
        ]
    )
    + "\n",
    encoding="utf-8",
)

for name, bench, pipes in [
    (
        "Dedup",
        Benchmark.Dedup(dataset=gold("dedup_gold.jsonl")),
        {"minhash": Baseline(preprocess=PreprocessConfig(doc_dedup_method="minhash"))},
    ),
    (
        "Chunking",
        Benchmark.Chunking(dataset=gold("chunking_gold.jsonl", 200)),
        {"sentence": Baseline(preprocess=PreprocessConfig(chunk_method="sentence"))},
    ),
    (
        "Extraction",
        Benchmark.Extraction(dataset=gold("ner_gold.jsonl", 100)),
        {"spacy": Baseline()},
    ),
    (
        "Resolution",
        Benchmark.Resolution(dataset=gold("resolution_gold.jsonl")),
        {"string": Baseline()},
    ),
    (
        "Quality",
        Benchmark.Quality(dataset=gold("quality_gold.jsonl")),
        {"surface": Baseline()},
    ),
]:
    m = list(bench.run(pipelines=pipes).values())[0]
    print(f"{name:10}", "  ".join(f"{k}={v:.3f}" for k, v in m.items() if isinstance(v, float)))

m = list(
    Benchmark.RAG(dataset=RAG_GOLD)
    .run(
        pipelines={
            "surface": Baseline(
                preprocess=PreprocessConfig(
                    chunk_method="sentence",
                    doc_dedup_method="minhash",
                    chunk_dedup_method="minhash",
                )
            )
        },
        input_paths=[str(CORPUS)],
    )
    .values()
)[0]
print("RAG       ", "  ".join(f"{k}={v:.3f}" for k, v in m.items() if isinstance(v, float)))

## Reading the numbers

- **Whole-pipeline** — a single `overall_score` for the end-to-end run; `nodes`/`edges` describe the KG it built.
- **Stage benchmarks** — each isolates ONE stage against gold data, so a low score points at the weak stage:
  - `Dedup` — near-duplicate detection (DBLP-ACM gold). Case-sensitive methods score low *recall* — an honest finding, not a bug.
  - `Chunking` / `Extraction` — whether chunks keep gold entities intact / how well spaCy NER matches gold spans.
  - `Resolution` / `Quality` — the bundled gold is placeholder (T2D / TACRED are license-gated), so those numbers are only meaningful once you supply real gold via `dataset=` or `--dataset`.
  - `RAG` — retrieval from the KG the pipeline builds; the demo answers are drawn from the corpus, so recall is high.

Swap `Baseline` for `Semantic()` (or any custom `Pipeline` subclass) in the cells above to compare pipelines.